# 01 — EDA: dataset reconciliation

Reconciles `PLAN.md §3` assumptions against what's actually on disk under `data/raw/`. Re-run after any change to the data folder.

Datasets surveyed:
- **CarDD** (`nasimetemadi/car-damage-detection`) — damage-type training corpus.
- **comprehensive-car-damage-detection** — front/rear × condition auxiliary head.
- **iaai-dataset** — metadata distributions (cost paywalled in this sample).
- **Stanford Cars 196** — make/model/year identifier training (Phase 1.5).

See [../CITATIONS.md](../CITATIONS.md) for full attributions.

In [1]:
import os, sys
from pathlib import Path
# Run notebook from repo root so relative data paths in loaders resolve.
if Path.cwd().name == 'notebooks':
    os.chdir('..')
sys.path.insert(0, 'src')

from collections import Counter
from itertools import islice

from ccdp.data import DAMAGE_TYPES
from ccdp.data.loaders import (
    CARDD_ROOT, COMPREHENSIVE_ROOT, IAAI_ROOT,
    iter_cardd, iter_comprehensive, iter_iaai,
)

RAW = Path('data/raw')
print('cwd:', Path.cwd())
print('raw root exists:', RAW.exists())
for d in sorted(RAW.iterdir()) if RAW.exists() else []:
    if d.is_dir() and not d.name.startswith('.'):
        n_imgs = sum(1 for p in d.rglob('*.jp*g')) + sum(1 for p in d.rglob('*.png'))
        print(f'  {d.name:50s} images={n_imgs}')

cwd: /Users/abhishekroy/Cursor/CarCrashFixAmountPredictor
raw root exists: True
  car-damage-detection                               images=16000
  car-damage-detection-and-cost-estimation           images=128
  comprehensive-car-damage-detection                 images=2300
  iaai-dataset                                       images=0
  stanford-cars-dataset                              images=16185


## CarDD — damage-type training corpus

Six canonical damage TYPES. COCO segmentation; converted to YOLO-format bboxes at load time. Used by both Variant A (multi-label) and Variant B (YOLOv8 detection).

In [2]:
cardd_records = []
type_counts = Counter()
bbox_counts = Counter()
for r in iter_cardd():
    cardd_records.append(r)
    type_counts.update(r.damage_types)
    for b in r.bboxes:
        bbox_counts[b.damage_type] += 1

print(f'CarDD images: {len(cardd_records)}')
print('image-level type counts:')
for t in sorted(type_counts, key=type_counts.get, reverse=True):
    print(f'  {t:20s} {type_counts[t]:5d}')
print('bbox-level type counts:')
for t in sorted(bbox_counts, key=bbox_counts.get, reverse=True):
    print(f'  {t:20s} {bbox_counts[t]:5d}')
print(f'total bboxes: {sum(bbox_counts.values())}')
print(f'unknown categories encountered (should be 0): {set(type_counts) - set(DAMAGE_TYPES)}')

CarDD images: 4000
image-level type counts:
  scratch               2121
  dent                  1751
  lamp_broken            693
  glass_shatter          674
  crack                  604
  tire_flat              309
bbox-level type counts:
  scratch               3595
  dent                  2543
  crack                  898
  lamp_broken            704
  glass_shatter          681
  tire_flat              319
total bboxes: 8740
unknown categories encountered (should be 0): set()


## comprehensive — front/rear × condition

Folder-encoded labels. Used as the auxiliary location/condition head.

In [3]:
comp_loc = Counter()
comp_cond = Counter()
comp_n = 0
for r in iter_comprehensive():
    comp_n += 1
    comp_loc[r.damage_location] += 1
    comp_cond[r.damage_condition] += 1

print(f'comprehensive images: {comp_n}')
print('location:', dict(comp_loc))
print('condition:', dict(comp_cond))

comprehensive images: 2300
location: {'front': 1400, 'rear': 900}
condition: {'normal': 800, 'crushed': 700, 'breakage': 800}


## iaai — metadata distributions only

Cost columns are masked `"[PREMIUM]"` in the free sample (12,353 rows). Free fields: year, make, model, bodyStyle, vehicleClass, exteriorColor, mileage, primaryDamage. Used as the car-metadata distribution source for the reference table.

In [4]:
iaai_n = 0
iaai_make = Counter()
iaai_body = Counter()
iaai_loc = Counter()
iaai_year = []
iaai_has_cost = 0
for r in iter_iaai():
    iaai_n += 1
    if r.make: iaai_make[r.make] += 1
    iaai_body[r.body_type] += 1
    iaai_loc[r.damage_location] += 1
    if r.year: iaai_year.append(r.year)
    if r.cost_usd is not None: iaai_has_cost += 1

print(f'iaai rows: {iaai_n}')
print(f'rows with usable cost_usd: {iaai_has_cost}  (expected 0 in free sample)')
if iaai_year:
    yrs = sorted(iaai_year)
    print(f'year range: {yrs[0]}-{yrs[-1]}  median={yrs[len(yrs)//2]}')
print('top 10 makes:')
for m, c in iaai_make.most_common(10):
    print(f'  {m:20s} {c:5d}')
print('body_type:', dict(iaai_body))
print('damage_location (front/rear/unknown):', dict(iaai_loc))

iaai rows: 12353
rows with usable cost_usd: 0  (expected 0 in free sample)
year range: 1965-2026  median=2014
top 10 makes:
  ford                  2635
  toyota                1936
  honda                 1308
  nissan                1012
  chevrolet              962
  hyundai                600
  kia                    533
  dodge                  463
  volkswagen             309
  ram                    275
body_type: {'sedan': 7512, 'hatchback': 1170, 'unknown': 128, 'pickup': 2644, 'coupe': 535, 'suv': 98, 'wagon': 112, 'convertible': 154}
damage_location (front/rear/unknown): {'front': 5079, 'unknown': 5646, 'rear': 1628}


## Stanford Cars — make/model/year (Phase 1.5)

Sanity-check the on-disk layout. We don't have a loader for Stanford Cars yet; that arrives with Phase 1.5.

In [5]:
sc_root = Path('data/raw/stanford-cars-dataset')
if sc_root.exists():
    train = sc_root / 'cars_train' / 'cars_train'
    test  = sc_root / 'cars_test'  / 'cars_test'
    devkit = sc_root / 'car_devkit' / 'devkit'
    print(f'train images: {len(list(train.glob("*.jpg")))}')
    print(f'test images:  {len(list(test.glob("*.jpg")))}')
    if devkit.exists():
        print('devkit files:')
        for p in sorted(devkit.iterdir()):
            print(f'  {p.name}')
else:
    print('stanford-cars-dataset not on disk yet')

train images: 8144
test images:  8041
devkit files:
  README.txt
  cars_meta.mat
  cars_test_annos.mat
  cars_train_annos.mat
  eval_train.m
  train_perfect_preds.txt


## Reference table (built from iaai metadata)

Run `ccdp data build-reference-table` from the CLI first; this cell just reads it back.

In [6]:
from ccdp.identification import reference_table as reftab
try:
    rep = reftab.coverage_report()
    print(rep)
    df = reftab.load()
    print('\nfirst 10 rows:')
    print(df.head(10).to_string())
except FileNotFoundError as e:
    print(e)
    print('Run `ccdp data build-reference-table` first.')

{'rows': 811, 'unique_makes': 43, 'unique_models': 260, 'year_range': [1978, 2026], 'body_type_counts': {'sedan': 464, 'pickup': 138, 'hatchback': 114, 'coupe': 33, 'unknown': 26, 'convertible': 18, 'wagon': 15, 'suv': 3}, 'segment_counts': {'mid': 435, 'luxury': 193, 'economy': 183}, 'total_samples': 2000, 'median_avg_cost_usd': nan}

first 10 rows:
         make   model  year body_type segment avg_cost_usd  n_samples datasets
0       acura      rl  2004     sedan  luxury         None          1     iaai
1       acura      tl  2005     sedan  luxury         None          1     iaai
2       acura      tl  2006     sedan  luxury         None          1     iaai
3       acura      tl  2009     sedan  luxury         None          1     iaai
4       acura     tlx  2015     sedan  luxury         None          1     iaai
5       acura     tlx  2021     sedan  luxury         None          1     iaai
6       acura     tsx  2004     sedan  luxury         None          1     iaai
7       acura  

## Summary

| Source | Rows / images | Trainable labels | Cost data |
|---|---|---|---|
| CarDD | ~4000 images with COCO bboxes | 6 damage TYPES | — |
| comprehensive | ~2300 images | front/rear × condition | — |
| iaai (free) | 12,353 metadata rows | — | paywalled |
| Stanford Cars | 16,185 images | 196 make/model/year classes | — |

Cost supervision comes from the **versioned parts-cost catalog** (transparent rule-based pricing); see `PLAN.md §3` honesty statement and the cost-calibration design in `progress/phase_0_scaffold.md`.